# Week 32, Day 3: NamedTuple

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w032d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: int
    y: int = 0

p = Point(3, 4)
print(p, p.x, p[1], len(p))
x, y = p
print(x + y, p == (3, 4), Point(3) == Point(3, 0))

In [ ]:
from typing import NamedTuple

class Member(NamedTuple):
    """A club member's details."""
    name: str
    age: int
    town: str = "Leeds"

    @property
    def is_junior(self):
        return self.age < 18

    def label(self):
        return f"{self.name} ({self.town})"

m = Member("Ada", 31)
print(m, m.label(), m.is_junior, Member("Tobi", 12, "Ibadan").is_junior)
try:
    m.age = 32
except AttributeError as err:
    print("AttributeError:", err)

In [ ]:
from typing import NamedTuple

class Stop(NamedTuple):
    name: str
    minutes: int

route = [Stop("CMS", 70), Stop("Ikorodu", 0), Stop("Ketu", 25)]

for name, minutes in sorted(route, key=lambda s: s.minutes):   # unpacking in a loop
    print(f"{minutes:>3} min  {name}")

ketu = route[2]
print(ketu._replace(minutes=30), ketu._asdict())
print(Stop._fields, Stop._make(["Obalende", 65]))
times = {Stop("Ketu", 25): "busy"}          # hashable: usable as a key
print(times[Stop("Ketu", 25)], max(route, key=lambda s: s.minutes).name)

In [ ]:
from typing import NamedTuple

class Summary(NamedTuple):
    low: float
    high: float
    mean: float

def summarise(values):
    if not values:
        raise ValueError("no values")
    return Summary(min(values), max(values), round(sum(values) / len(values), 2))

temps = [14.5, 22.1, 30.4, 18.0]
s = summarise(temps)
print(s.high - s.low, s)
low, high, mean = summarise(temps)
print(mean)

In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: int
    y: int

class Size(NamedTuple):
    width: int
    height: int

p = Point(3, 4)
print(p == (3, 4), p == Size(3, 4))       # equal to any tuple with the same values
print(p * 2, p + (5,))                   # tuple repetition and joining, not maths
print(len(p), list(p), sum(p))           # it's a sequence of its values
print(sorted([Point(2, 1), Point(1, 5)]))  # ordered field by field, like tuples

> **Key idea.** `class P(NamedTuple):` with annotated fields makes an immutable tuple type with names: `__init__`, `__repr__`, `==`, ordering and hashing come from the tuple, and you can add methods, properties and a docstring, but not `__init__`. Named tuples unpack, index, sort and hash like tuples, and have `_replace`, `_asdict`, `_fields` and `_make`; they make clear return values. Because they're tuples, they're equal to any tuple with the same values, `*` and `+` repeat and join, and their immutability is shallow. Choose a NamedTuple for tuple-like records, a dataclass (frozen for values) for things with their own meaning.

### Exercise: Returning a summary

Write a NamedTuple `Stats` with fields `count`, `low`, `high` and `mean`, and a function `stats(values)` that returns one: the number of values, the smallest, the largest, and the mean rounded to 2 places. An empty list raises `ValueError`. Give `Stats` a property `spread`, the difference between `high` and `low`.

In [ ]:
from typing import NamedTuple

___

def stats(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stats():\n    \"a summary\"\n    S = ns[\"Stats\"]\n    s = stats([4, 8, 15, 16, 23, 42])\n    assert s == (6, 4, 42, 18.0) and s.mean == 18.0 and s.spread == 38 and isinstance(s, tuple)\n    assert repr(stats([2.5])) == \"Stats(count=1, low=2.5, high=2.5, mean=2.5)\" and stats([1, 2]).mean == 1.5\n    n, lo, hi, mean = stats([3, 1, 2])\n    assert (n, lo, hi, mean) == (3, 1, 3, 2.0) and S._fields == (\"count\", \"low\", \"high\", \"mean\")\n    try:\n        stats([])\n        assert False, \"an empty list should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Colours as named tuples

Write a NamedTuple `RGB` with fields `r`, `g` and `b`. Add:

- a property `hex`, like `"#ff8000"`;
- a class method `from_hex(text)` reading `"#ff8000"` or `"FF8000"`, raising `ValueError` for anything else (it must be six hexadecimal digits, after an optional `#`);
- `brighter(amount)`, returning a **new** colour with `amount` added to each channel, kept between 0 and 255 (a negative amount makes it darker).

In [ ]:
from typing import NamedTuple

class RGB(NamedTuple):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rgb():\n    \"colours\"\n    R = ns[\"RGB\"]\n    c = R(255, 128, 0)\n    assert c.hex == \"#ff8000\" and R.from_hex(\"#FF8000\") == c and R.from_hex(\"0a0b0c\") == (10, 11, 12)\n    assert c.brighter(10) == R(255, 138, 10) and c.brighter(-200) == R(55, 0, 0) and c == R(255, 128, 0)\n    assert type(c.brighter(1)).__name__ == \"RGB\"\n    for bad in [\"#ff800\", \"#gg8000\", \"ff80000\", \"\", \"#ff 800\"]:\n        try:\n            R.from_hex(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Rows from a file

`stations.csv` has a header line and one station per line. Write a NamedTuple `Station` with fields `name`, `code` and `zone` (an `int`), and `load(path)`, which reads the file with `csv.reader` and returns a list of stations. The header must match the field names exactly, otherwise raise `ValueError`. Then write `by_zone(stations)`: a dictionary from zone to a sorted list of station codes.

In [ ]:
# Run this cell first: it creates the files used below.
with open('stations.csv', 'w', encoding='utf-8') as f:
    f.write('name,code,zone\nIkeja,IKJ,2\nYaba,YAB,1\nAgege,AGG,3\nOshodi,OSH,2\n')

In [ ]:
import csv
from typing import NamedTuple

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_load():\n    \"loading\"\n    S = ns[\"Station\"]\n    rows = load(\"stations.csv\")\n    assert rows[0] == S(\"Ikeja\", \"IKJ\", 2) and len(rows) == 4 and rows[1].zone == 1 and isinstance(rows[3].zone, int)\n    assert by_zone(rows) == {2: [\"IKJ\", \"OSH\"], 1: [\"YAB\"], 3: [\"AGG\"]}\n    write_file(\"bad.csv\", \"name,zone,code\\nX,1,XX\\n\")\n    try:\n        load(\"bad.csv\")\n        assert False, \"a header that doesn't match should raise ValueError\"\n    except ValueError:\n        pass\n    write_file(\"empty.csv\", \"name,code,zone\\n\")\n    assert load(\"empty.csv\") == []")

### Exercise: Positions in a set

Write a NamedTuple `Pos` with fields `row` and `col`, and a method `neighbours(rows, cols)` returning the positions up, down, left and right of it that are inside a grid of that size (in that order). Then write `reachable(grid, start)`: `grid` is a list of strings where `#` is a wall; return the **set** of `Pos` that can be reached from `start` by moving up, down, left and right without crossing walls. Starting on a wall reaches nothing.

In [ ]:
from typing import NamedTuple

class Pos(NamedTuple):
    ___

def reachable(grid, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pos():\n    \"neighbours\"\n    P = ns[\"Pos\"]\n    assert P(0, 0).neighbours(3, 3) == [P(1, 0), P(0, 1)]\n    assert P(1, 1).neighbours(3, 3) == [P(0, 1), P(2, 1), P(1, 0), P(1, 2)]\ndef test_reachable():\n    \"reaching\"\n    P = ns[\"Pos\"]\n    grid = [\"..#.\",\n            \".##.\",\n            \"...#\"]\n    got = reachable(grid, P(0, 0))\n    assert got == {P(0, 0), P(0, 1), P(1, 0), P(2, 0), P(2, 1), P(2, 2)}, f\"got {sorted(got)}\"\n    assert reachable(grid, P(0, 3)) == {P(0, 3), P(1, 3)} and reachable(grid, P(0, 2)) == set()\n    assert len(reachable([\".\" * 30] * 30, P(0, 0))) == 900")

### Bug hunt: Doubling a vector

`double` should give `Vec(x=6, y=8)`, but it prints `(3, 4, 3, 4)`. Fix `double` (keep `Vec` a NamedTuple).

In [ ]:
from typing import NamedTuple

class Vec(NamedTuple):
    x: int
    y: int

def double(v):
    return v * 2

print(double(Vec(3, 4)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"doubled\"\n    expect_output(\"Vec(x=6, y=8)\")")

### Bug hunt: Moving a booking

Moving a booking crashes with `AttributeError`, because named tuples can't be changed. Fix `move` so it returns an updated booking, and the program prints `Booking(room='B2', day='Tue', slot=3)`.

In [ ]:
from typing import NamedTuple

class Booking(NamedTuple):
    room: str
    day: str
    slot: int

def move(booking, new_slot):
    booking.slot = new_slot
    return booking

print(move(Booking("B2", "Tue", 1), 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"moved\"\n    expect_output(\"Booking(room='B2', day='Tue', slot=3)\")")

### Bug hunt: Two units, one key

A weather log keeps readings in Celsius and Fahrenheit, but `20°C` and `20°F` count as the same reading, so this prints `1` instead of `2`. Make each a frozen dataclass instead of a NamedTuple, so different kinds are never equal.

In [ ]:
from dataclasses import dataclass
from typing import NamedTuple

class Celsius(NamedTuple):
    degrees: float

class Fahrenheit(NamedTuple):
    degrees: float

readings = {Celsius(20), Fahrenheit(20)}
print(len(readings))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"two readings\"\n    expect_output(\"2\")\n    import dataclasses\n    assert dataclasses.is_dataclass(ns[\"Celsius\"]) and dataclasses.is_dataclass(ns[\"Fahrenheit\"])\n    assert ns[\"Celsius\"](5) == ns[\"Celsius\"](5)")

### Puzzle: A named tuple with a default

Arrange the lines into a NamedTuple for a parcel, whose service defaults to `"standard"`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from typing import NamedTuple

class P(NamedTuple):
    a: int
    b: int = 2

class Q(NamedTuple):
    c: int
    d: int

p = ___   # P(1)
q = ___   # P(1) == Q(1, 2)
r = ___   # P(5, 0) < P(4, 9)
s = ___   # P(1, 2) + P(3)
t = ___   # P(7)._replace(a=0)
u = ___   # P._fields

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    P = ns[\"P\"]\n    assert p == (1, 2) and q is True and r is False, \"p to r\"\n    assert s == (1, 2, 3, 2) and t == (0, 2) and u == (\"a\", \"b\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: summarising sales

`sales.csv` holds one sale per line: `day,shop,item,qty,pence` (the price of **one** item). Write:

- a NamedTuple `Sale` with those fields (`qty` and `pence` as `int`s) and a property `total` (qty times pence);
- `load_sales(path)`: read the file (header first, which must be exactly those field names) into a list of `Sale`. Raise `ValueError` mentioning the line number (counting the header as line 1) if a line doesn't have five fields, or `qty` or `pence` isn't a whole number of 1 or more for `qty`, 0 or more for `pence`;
- a NamedTuple `ShopSummary` with fields `shop`, `sales` (how many lines), `revenue` (the sum of totals) and `best_item` (the item that brought in the most revenue at that shop, the first alphabetically if tied);
- `summarise(sales)`: a list of `ShopSummary`, one per shop, highest revenue first, ties by shop name A to Z.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sales.csv', 'w', encoding='utf-8') as f:
    f.write('day,shop,item,qty,pence\n2026-11-02,Ikeja,suya,3,500\n2026-11-02,Yaba,zobo,10,150\n2026-11-02,Ikeja,zobo,4,150\n2026-11-03,Yaba,puff-puff,20,50\n2026-11-03,Lekki,suya,1,500\n2026-11-03,Ikeja,suya,2,500\n')

In [ ]:
import csv
from typing import NamedTuple

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_load():\n    \"loading\"\n    S = ns[\"Sale\"]\n    sales = load_sales(\"sales.csv\")\n    assert len(sales) == 6 and sales[0] == S(\"2026-11-02\", \"Ikeja\", \"suya\", 3, 500) and sales[0].total == 1500\n    for bad, line in [(\"day,shop,item,qty,pence\\nx,y,z,1\\n\", 2), (\"day,shop,item,qty,pence\\na,b,c,1,5\\na,b,c,0,5\\n\", 3),\n                      (\"day,shop,item,qty,pence\\na,b,c,2,-5\\n\", 2), (\"day,shop,item,qty,pence\\na,b,c,two,5\\n\", 2)]:\n        write_file(\"bad.csv\", bad)\n        try:\n            load_sales(\"bad.csv\")\n            assert False, f\"line {line} should raise ValueError\"\n        except ValueError as err:\n            assert str(line) in str(err), f\"the message should mention line {line}: {err}\"\n    write_file(\"bad.csv\", \"shop,day,item,qty,pence\\n\")\n    try:\n        load_sales(\"bad.csv\")\n        assert False, \"a wrong header should raise ValueError\"\n    except ValueError:\n        pass\ndef test_summary():\n    \"summaries\"\n    SS = ns[\"ShopSummary\"]\n    got = summarise(load_sales(\"sales.csv\"))\n    assert got == [SS(\"Ikeja\", 3, 3100, \"suya\"), SS(\"Yaba\", 2, 2500, \"zobo\"), SS(\"Lekki\", 1, 500, \"suya\")], f\"got {got}\"\n    S = ns[\"Sale\"]\n    tie = [S(\"d\", \"A\", \"b\", 1, 100), S(\"d\", \"A\", \"a\", 1, 100), S(\"d\", \"B\", \"x\", 2, 100)]\n    assert summarise(tie) == [SS(\"A\", 2, 200, \"a\"), SS(\"B\", 1, 200, \"x\")] and summarise([]) == []")

## Warm-up

`Point` looks like yesterday's dataclasses (fields with annotations and a default), but its objects are **tuples**: they have a length, can be indexed and unpacked, and are equal to a plain tuple holding the same values. Today is about when that's exactly what you want, and when it isn't.

## Class syntax for named tuples

Week 20 made named tuples with a function call:

```python
from collections import namedtuple
Member = namedtuple("Member", ["name", "age", "town"])
```

`typing.NamedTuple` makes the same thing with a class statement, which reads better and allows more:

- Fields are written like a dataclass's: `name: type`, with defaults after the fields without them. As with dataclasses, the annotations aren't checked.
- You get `__init__`, a readable `__repr__`, and `==`, ordering and hashing **from the tuple**.
- Objects are **immutable**: fields can't be assigned. (They're tuples, after all.)
- You can add methods, properties, class methods and a docstring. What you can't do is write your own `__init__`: Python refuses, because a tuple's values are fixed before `__init__` would run. For checking values, use a class method that checks and then makes the tuple, or use a frozen dataclass.

## Still a tuple

Everything a tuple can do, a named tuple can do, plus a few helpers whose names start with an underscore (so they can't clash with your field names; they're public despite the underscore):

| Helper | Gives |
|---|---|
| `p._replace(field=value)` | a new tuple with some fields changed |
| `p._asdict()` | a dictionary of the fields |
| `Cls._fields` | the field names, as a tuple |
| `Cls._make(iterable)` | a new tuple from any iterable, such as a row from `csv` |
| `Cls._field_defaults` | a dictionary of the defaults |

Named tuples shine as **return values**. A function that returns several results as a plain tuple makes callers remember the order; a named tuple lets them use names or unpack, as they prefer, and costs nothing extra:

Python's own library does the same: `os.stat()`, `time.localtime()` and `sys.version_info` all return named tuples.

## Where tuples surprise

Being a tuple has costs. Each of these is fine for a simple record, and a real bug for a value with its own meaning:

- A `Point` equals a `Size` with the same numbers, and hashes the same, so they collide in sets and dictionaries.
- `p * 2` repeats the tuple. If you wanted a scaled point, you need a method (or a class with `__mul__`).
- Named tuples have a length and can be iterated, which may let them slip into code that expects a list.
- "Immutable" is **shallow**: a named tuple holding a list can't swap the list for another, but the list itself can change, and then the tuple is unhashable.

Dataclasses (`frozen=True`) don't have these problems: they're only equal to objects of their own class, and have no length, no `+` and no indexing unless you add them.

## Choosing a record type

Here's a way to choose, from the simplest to the most capable:

| Use | When |
|---|---|
| a plain `tuple` | two or three values, used briefly, whose order is obvious (`(x, y)`, the pairs from `dict.items()`) |
| a `NamedTuple` | an immutable record with named fields that it's fine to treat as a tuple: return values, rows from a file, small records used as keys |
| a `@dataclass` | a record with its own identity and behaviour, which may change, may need checks in `__post_init__`, and shouldn't be confused with a tuple |
| `@dataclass(frozen=True)` | an immutable **value** with its own meaning (money, versions, coordinates): hashable, but only equal to its own kind |
| a hand-written class | complicated rules about attributes, an `__init__` that does more than store arguments, or heavy use of the data model |
| a `dict` | data whose keys aren't known in advance, such as JSON you're just passing through |

Named tuples also take very little memory (no per-object dictionary, like slotted classes), which matters when you make very many of them, for example reading a large CSV file.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a function you've written that returns a tuple of two or three results, and make it return a NamedTuple instead; check that the code calling it still works unchanged. Tomorrow: abstract base classes, which make "every child must write this method" a rule Python enforces.